In [25]:
import numpy as np
import pandas as pd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, cross_val_predict, GridSearchCV
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay, confusion_matrix, mean_absolute_error, mean_squared_error, r2_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn import svm
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline, Pipeline

In [31]:
filename= 'datos_limpios.csv'
data= pd.read_csv(filename)
df=pd.DataFrame(data)
df.head()

,host_listings_count,host_total_listings_count,latitude,longitude,accommodates,bathrooms,bedrooms,beds,price,minimum_nights,...,room_type_Hotel room,room_type_Private room,room_type_Shared room,calendar_last_scraped_2025-09-16,calendar_last_scraped_2025-09-17,calendar_last_scraped_2025-09-22,calendar_last_scraped_2025-09-23,calendar_last_scraped_2025-09-24,calendar_last_scraped_2025-09-25,calendar_last_scraped_2025-09-30
0,1.0,2.0,30.26057,-97.73441,3,1.0,1.0,2.0,97.0,2,...,0,0,0,0,1,0,0,0,0,0
1,1.0,2.0,30.26034,-97.76487,2,1.0,1.0,2.0,160.0,3,...,0,0,0,0,1,0,0,0,0,0
2,1.0,1.0,30.23466,-97.73682,2,1.0,1.0,1.0,38.0,4,...,0,0,0,0,1,0,0,0,0,0
3,2.0,2.0,30.26098,-97.73072,3,2.0,2.0,2.0,145.0,15,...,0,0,0,0,1,0,0,0,0,0
4,1.0,1.0,30.23614,-97.73225,2,1.0,1.0,1.0,58.0,30,...,0,0,0,1,0,0,0,0,0,0


In [32]:
df = df.fillna(df.median(numeric_only=True))

In [33]:
#Se procede a la separación de las variables de esta manera,
#dado que según la instruccion del inciso 11, se ha de hacer la regresión directamente respecto al precio
y = df["price"]

#Variables predictoras
X = df.drop(columns=["price"])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [34]:
columnas=X_train.columns
scaler = MinMaxScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [35]:
X_train=pd.DataFrame(X_train, columns=columnas)
X_test=pd.DataFrame(X_test, columns=columnas)

In [36]:
Q1 = X_train.quantile(0.25)
Q3 = X_train.quantile(0.75)
IQR = Q3 - Q1
inf = Q1 - 3 * IQR
sup = Q3 + 3 * IQR

PROBLEMA 10: tabla comparativa de los modelos

In [37]:
tabla=pd.DataFrame({
    "Modelo": ["Arbol de decisión","Random Forest","Naive Bayes","KNN","Regresión Logística"],
    "Predicciones correctas": [9099, 12524,7639,10358,12896],
    "Predicciones incorrectas":[4801,4318,7080,4361,1814 ]
})

In [38]:
tabla

,Modelo,Predicciones correctas,Predicciones incorrectas
0,Arbol de decisión,9099,4801
1,Random Forest,12524,4318
2,Naive Bayes,7639,7080
3,KNN,10358,4361
4,Regresión Logística,12896,1814


Es posible observar que la Regresión Logística es aquel modelo que tiene mayor número de aciertos y menor cantidad de errores. No obstante, como se ha mencionado en ese laboratorio, este modelo daba indicios de cierto sobreajuste. Le sigue el modelo de Random Forest, que en laboratorios anteriores se determinó que era el mejor modelo, ya que no estaba sobreajustado del todo y dada una cantidad de predicciones correctas lo suficientemente alta. Se observa que el modelo de KNN le seguiría a este ya que tiene una cantidad de aciertos elevada, pero la cantidad de predicciones incorrectas no es tan baja como para considerarlo sobreajuste. El modelo de árbol de decisión es cuarto en ser el mejor, ya que el número de predicciones incorrectas es casi la mitad en proporción de aquellas correctas. Finalmente se encuentra el modelo de Naive Bayes, el cual se puede observar que la cantidad de predicciones erradas es ligeramente mayor que la cantidad de predicciones correctas.

PROBLEMA 11: Modelo de Regresión

In [40]:
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", svm.SVR())
])

param_grid = {
    'svr__kernel': ['linear', 'rbf', 'sigmoid'],
    'svr__C': [0.5, 1, 10],
    'svr__gamma': ['scale', 0.1, 1]
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=2,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

grid.fit(X_train, y_train)

KeyboardInterrupt: 

In [29]:
print("Mejores parámetros:")
print(grid.best_params_)

Mejores parámetros:


AttributeError: 'GridSearchCV' object has no attribute 'best_params_'

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n=== Métricas ===")
print("MSE:", mse)
print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)